In [90]:
from bs4 import BeautifulSoup
import json

In [91]:
def win_loss(goalsA: int, goalsB: int, positionA: int, positionB: int, isHome: bool, isWin: bool, mult1: float = 0.25, mult2: float = 1.5, base: int = 5):
    # print(goalsA, goalsB, positionA, positionB, isHome, isWin)
    if isWin:
        if isHome:
            return base + (positionA - positionB)*mult1
        else:
            return mult2*(base + (positionA - positionB)*mult1)
    else:
        if isHome:
            return mult2*(-base + (positionA - positionB)*mult1)
        else:
            return -base + (positionA - positionB)*mult1

In [92]:
def draw(goalsA: int, goalsB: int, positionA: int, positionB: int, isHome: bool, mult1: float = 0.5, mult2: float = 0.75):
    # print(goalsA, goalsB, positionA, positionB, isHome)
    if positionA - positionB < 0 and isHome:
        return (positionA - positionB)*mult2
    if positionA - positionB < 0:
        return (positionA - positionB)*mult1
    if positionA - positionB > 0 and not isHome:
        return (positionA - positionB)*mult2
    if positionA - positionB > 0:
        return (positionA - positionB)*mult1

In [93]:
with open('Data/SerieA_27_10_2024_15_41.html', 'r', encoding='utf-8') as f:
    matches = f.read()

soup = BeautifulSoup(matches, 'html.parser')

matchesDict = {}
for i, d in enumerate(soup.find_all('div', class_="event__match")):
    teamA = d.find('div', class_="event__homeParticipant")
    teamB = d.find('div', class_="event__awayParticipant")
    goalsA = d.find('div', class_="event__score--home")
    goalsB = d.find('div', class_="event__score--away")

    matchesDict[i] = {'teamA': teamA.text, 'teamB': teamB.text, 'goalsA': goalsA.text, 'goalsB': goalsB.text}

with open('Data/SerieA_27_10_2024_15_41.json', 'w', encoding='utf-8') as f:
    json.dump(matchesDict, f)

In [94]:
with open('Data/Table_SerieA_27_10_2024_15_41.html', 'r', encoding='utf-8') as f:
    matches = f.read()

soup = BeautifulSoup(matches, 'html.parser')

teamsDict = {}

for d in soup.find_all('div', class_="ui-table__row"):
    rank = d.find('div', class_="tableCellRank")
    team = d.find('a', class_="tableCellParticipant__name")

    teamsDict[team.text] = int(rank.text.replace(".", ""))

with open('Data/Table_SerieA_27_10_2024_15_41.json', 'w', encoding='utf-8') as f:
    json.dump(teamsDict, f)

In [95]:
def getMatchesForTeamName(teamName: str, isFirstPassed = False):
    teamToBeTested = teamName
    teamData = []

    isFirstPassed = isFirstPassed
    for m in matchesDict.values():
        if m["teamA"] == teamToBeTested or m["teamB"] == teamToBeTested:
            if not isFirstPassed:
                isFirstPassed = True
                continue
            isWin: bool = None
            opponent = ""
            if m['goalsA'] != m['goalsB']:
                if (m['teamA'] == teamToBeTested and m['goalsA'] > m['goalsB']) or (m['teamB'] == teamToBeTested and m['goalsA'] < m['goalsB']):
                    isWin = True
                else:
                    isWin = False
            if m["teamA"] == teamToBeTested:
                opponent = m["teamB"]
            else:
                opponent = m["teamA"]
            teamData.append([int(m['goalsA']), int(m['goalsB']), teamsDict[opponent], m['teamA'] == teamToBeTested, isWin])
        if len(teamData) == 5:
            break

    return teamData

In [96]:
def getResults(team1: str, team2: str):
    team1 = "Genoa"
    team2 = "Bologna"

    matches = getMatchesForTeamName(team1)
    position = teamsDict[team1]
    result = 0
    # print(matches)
    for m in matches:
        if m[0] == m[1]:
            result = result + draw(m[0], m[1], position, m[2], m[3])
        else:
            result = result + win_loss(m[0], m[1], position, m[2], m[3], m[4])

    print("!!!!!!!!")

    matches2 = getMatchesForTeamName(team2)
    position2 = teamsDict[team2]
    result2 = 0
    for m in matches2:
        if m[0] == m[1]:
            result2 = result2 + draw(m[0], m[1], position2, m[2], m[3])
        else:
            result2 = result2 + win_loss(m[0], m[1], position2, m[2], m[3], m[4])
    
    return (result, result2)

In [ ]:
# win_loss_mult1 = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.35, 1.4, 1.45, 1.5, 1.55, 1.6, 1.65, 1.7, 1.75, 1.8, 1.85, 1.9, 1.95, 2.0]
# win_loss_mult2 = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.35, 1.4, 1.45, 1.5, 1.55, 1.6, 1.65, 1.7, 1.75, 1.8, 1.85, 1.9, 1.95, 2.0]

win_loss_mult1 = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0,1.1,1.2,1.3,1.4,1.5,1.6,1.7,1.8,1.9,2.0]
win_loss_mult2 = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0,1.1,1.2,1.3,1.4,1.5,1.6,1.7,1.8,1.9,2.0]

win_loss_base = [1,2,3,4,5,6,7,8,9,10]
# draw_mult1 = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.35, 1.4, 1.45, 1.5, 1.55, 1.6, 1.65, 1.7, 1.75, 1.8, 1.85, 1.9, 1.95, 2.0]
# draw_mult2 = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.35, 1.4, 1.45, 1.5, 1.55, 1.6, 1.65, 1.7, 1.75, 1.8, 1.85, 1.9, 1.95, 2.0]

draw_mult1 = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0,1.1,1.2,1.3,1.4,1.5,1.6,1.7,1.8,1.9,2.0]
draw_mult2 = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0,1.1,1.2,1.3,1.4,1.5,1.6,1.7,1.8,1.9,2.0]

matchesDict2 = {
    "0": {
        "teamA": "Verona",
        "teamB": "Monza",
        "goalsA": "0",
        "goalsB": "3"
    },
    "1": {
        "teamA": "AS Roma",
        "teamB": "Inter",
        "goalsA": "0",
        "goalsB": "1"
    },
    "2": {
        "teamA": "Cagliari",
        "teamB": "Torino",
        "goalsA": "3",
        "goalsB": "2"
    },
    "3": {
        "teamA": "Lecce",
        "teamB": "Fiorentina",
        "goalsA": "0",
        "goalsB": "6"
    },
    "4": {
        "teamA": "Venezia",
        "teamB": "Atalanta",
        "goalsA": "0",
        "goalsB": "2"
    },
    "5": {
        "teamA": "Empoli",
        "teamB": "Napoli",
        "goalsA": "0",
        "goalsB": "1"
    },
    "6": {
        "teamA": "Juventus",
        "teamB": "Lazio",
        "goalsA": "1",
        "goalsB": "0"
    },
    "7": {
        "teamA": "AC Milan",
        "teamB": "Udinese",
        "goalsA": "1",
        "goalsB": "0"
    },
    "8": {
        "teamA": "Como",
        "teamB": "Parma",
        "goalsA": "1",
        "goalsB": "1"
    },
    "9": {
        "teamA": "Genoa",
        "teamB": "Bologna",
        "goalsA": "2",
        "goalsB": "2"
    },
    "10": {
        "teamA": "Fiorentina",
        "teamB": "AC Milan",
        "goalsA": "2",
        "goalsB": "1"
    },
}

r = {}


for wlm1 in win_loss_mult1:
    print("@@@@@")
    for wlm2 in win_loss_mult2:
        for wlb in win_loss_base:
            for dm1 in draw_mult1:
                for dm2 in draw_mult2:
                    points = 0
                    for match in matchesDict2.values():
                        team1 = match['teamA']
                        team2 = match['teamB']

                        matches = getMatchesForTeamName(team1)
                        position = teamsDict[team1]
                        result = 0
                        for m in matches:
                            if m[0] == m[1]:
                                result = result + draw(m[0], m[1], position, m[2], m[3], dm1, dm2)
                            else:
                                result = result + win_loss(m[0], m[1], position, m[2], m[3], m[4], wlm1, wlm2, wlb)

                        # print("!!!!!!!")

                        matches2 = getMatchesForTeamName(team2)
                        position2 = teamsDict[team2]
                        result2 = 0
                        for m in matches2:
                            if m[0] == m[1]:
                                result2 = result2 + draw(m[0], m[1], position2, m[2], m[3], dm1, dm2)
                            else:
                                result2 = result2 + win_loss(m[0], m[1], position2, m[2], m[3], m[4], wlm1, wlm2, wlb)

                        if abs(result - result2) <= 5 and match['goalsA'] == match['goalsB']:
                            points+=1
                        elif result > result2 and match['goalsA'] > match['goalsB']:
                            points+=1
                        elif result < result2 and match['goalsA'] < match['goalsB']:
                            points+=1
                    r["{0} {1} {2} {3} {4}".format(wlm1,wlm2,wlb,dm1,dm2)] = points

In [107]:
with open('Data/wyniki.json', 'w', encoding='utf-8') as f:
    json.dump(r, f)

In [113]:
data_best = []

for rd in r:
    if r[rd] >= 8:
        data_best.append(rd)

In [114]:
import pandas as pd

data = [list(map(float, row.split())) for row in data_best]

# Create a DataFrame
df = pd.DataFrame(data, columns=['Col1', 'Col2', 'Col3', 'Col4', 'Col5'])

# Count occurrences of each value in each column
value_counts = {col: df[col].value_counts() for col in df.columns}

# Display results
for col, counts in value_counts.items():
    print(f"Counts for {col}:")
    print(counts)
    print()

Counts for Col1:
Col1
0.2    252
0.1    174
0.3    161
0.4     70
0.5     36
0.6      1
Name: count, dtype: int64

Counts for Col2:
Col2
0.1    486
0.2    177
0.3     26
0.4      5
Name: count, dtype: int64

Counts for Col3:
Col3
10.0    139
9.0     128
8.0     110
7.0      90
6.0      73
5.0      65
4.0      55
3.0      22
2.0      12
Name: count, dtype: int64

Counts for Col4:
Col4
0.1    168
0.2    134
0.3    109
0.4     84
0.5     54
0.6     43
0.7     39
0.8     33
0.9     22
1.0      8
Name: count, dtype: int64

Counts for Col5:
Col5
0.2    193
0.3    165
0.4    105
0.5     73
0.1     50
0.6     34
0.7     11
1.1     11
1.4      9
1.3      7
1.5      7
1.0      6
1.2      4
0.9      4
1.9      4
0.8      3
1.6      3
1.8      3
1.7      2
Name: count, dtype: int64

